# Chapter 25 – Worked examples

This notebook runs the examples of [Chapter 25](../index.md) live. The code lives in the modules next to it
(`map_elites_cells.py`, `repertoire.py`, `map_elites.py`); the cells only call them, so a change in a module is
a change here too. `small_grid_toop.py` needs ToOp and is not imported here: its output for the lab's step 2
is committed as `res_two_descriptors.json`.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository.

In [ ]:
import matplotlib

matplotlib.use('Agg')

import map_elites
import map_elites_cells
import matplotlib.pyplot as plt
import numpy as np
import repertoire

## From descriptors to a cell index

With the default descriptors, `n_cells_per_dim = (5, 45)`. `ravel_multi_index(..., mode='clip')` behaves like
the JAX call in ToOp's `get_cell_index`: 60 switching operations are clipped to column 44, 7 split substations
to row 4.

In [ ]:
print(map_elites_cells.cell_index(np.array([1, 7, 2]), np.array([60, 3, 44]), dims=(5, 45)))
print(
    'row 2 (exactly 2 splits) is cells',
    repertoire.cells_with_value((5, 45), 0, 2)[:3],
    '...',
    repertoire.cells_with_value((5, 45), 0, 2)[-1],
)
print(len(repertoire.clipped_cells((5, 45))), 'cells hold "at least" values')

## ToOp's repertoire from `res.json`

The lab's step 2 run (small grid, descriptors `split_subs` with 4 cells and `switching_distance` with 20 cells,
30 s on CPU). `summarize` only reports the cell elites that beat the unsplit grid, so the table is short and the
coverage computed from it is a lower bound.

In [ ]:
res = repertoire.load()
print(repertoire.descriptors(res), 'epochs:', res['iteration'])
print(repertoire.elites(res).to_string())
table = repertoire.grid(res)
print(
    'coverage',
    round(repertoire.coverage(table), 4),
    'QD-score',
    round(repertoire.qd_score(table, res['initial_fitness']), 3),
)
fig, ax = plt.subplots(figsize=(7, 2.5))
repertoire.heatmap(table, repertoire.descriptors(res), ax)
plt.close(fig)

## MAP-Elites in NumPy on the IEEE 30 line-switching problem

Descriptors: number of switched lines (0 to 3) and the maximum N-0 loading after switching in 5 bins. One seed,
300 evaluations; the archive prints the overload energy of every filled cell.

In [ ]:
case = map_elites.load_case30()
archive = map_elites.map_elites(case, seed=0, budget=300)
print(
    'coverage', round(archive.coverage, 2), 'QD-score', round(archive.qd_score, 1), 'best E', round(-archive.best, 3)
)
print(np.round(archive.table(), 1))

## A deceptive landscape

A wide hill of height 10 at $x = 20$ and a narrow peak of height 20 at $x = 85$, with a valley in between. The
elitist GA stays on the hill; MAP-Elites, with the tenth of the domain as descriptor, keeps an elite in every
valley cell and walks across.

In [ ]:
print('GA        :', [map_elites.ga_1d(seed) for seed in range(5)])
print('MAP-Elites:', [map_elites.map_elites_1d(seed) for seed in range(5)])